In [1]:
import pandas as pd

from collections import defaultdict

import json
import ast
import re

In [8]:
with open("../dataset/final_dataset/anon_cvs_en.json", encoding="utf-8") as f:
    data = json.load(f)

data

[{'headline': 'Experienced Senior IT SystemConsultant',
  'educationlevel': 8.0,
  'jobexperiences': 35.0,
  'mgrexperiences': 2.0,
  'countryid': 4,
  'cvid': 'ebf358ed252344of8d1dd3f4c8516cbf',
  'job_history': ['Cloud Infrastructure Specialist',
   'IT Administrator',
   'IT system consultant',
   'High-end operating consultant',
   'IT operation and support'],
  'educations': [{'type': 'education', 'name': 'Computer assistant'}],
  'languages': [{'code': 'a', 'level': 4},
   {'code': 'de', 'level': 2},
   {'code': 'da', 'level': 4}],
  'jobtitles': ['IT Administrator',
   'IT Manager',
   'IT consultant',
   'IT system consultant',
   'System administrator',
   'System Manager',
   'System consultant',
   'Senior Consultant',
   'Senior Adviser'],
  'keywords': ['Azure',
   'Operating optimisation',
   'Operating techniques',
   'Exchange server',
   'IT consultant',
   'IT Coordinator',
   'Management',
   'nutanix',
   'Problem Crusher',
   'SDM',
   'Serious and responsive',
   

In [9]:
for i in data:
    if i["cvid"] == "d296ec1752b8415b903e70f37b184854":
        print(i)

{'headline': 'Storage', 'educationlevel': 3.0, 'jobexperiences': 20.0, 'mgrexperiences': 0.0, 'countryid': 4, 'cvid': 'd296ec1752b8415b903e70f37b184854', 'job_history': ['Storage assistant', 'Postal officer', 'Production/stock', 'Storage assistant', 'Postal officer', 'Logistics Assistant', 'Warehouse staff', 'Vikar', 'Nature care', 'Property official/farmer', 'Property official/farmer', 'Property official/farmer', 'Property official/farmer', 'Package Item', 'Car Care', 'Vikar', 'IT Pendant', 'Netcafe/Teaching', 'Webmaster', 'Computer Support/ Data Guide', 'Webmaster', 'Teaching assistant', 'OCR operator'], 'educations': [{'type': 'education', 'name': 'Storage'}, {'type': 'education', 'name': 'Truck certificate B'}, {'type': 'education', 'name': 'Specialist work, soil and concrete'}, {'type': 'education', 'name': 'App development for mobiles and tablets'}, {'type': 'education', 'name': 'Managing and Maintaining a Microsoft Windows Server 2003 Environment'}, {'type': 'education', 'name':

In [2]:
df = pd.read_excel("./outputs/clean_outputs/triples_clean.xlsx").drop(["Unnamed: 0.1", "Unnamed: 0"], axis=1)

In [3]:
for col in df.columns:
    if "triples" in col:
        print(col, df[col].str.len().mean())

triples_qwen_structured 1200.8502456538172
triples_qwen_semi-structured 1376.4201625094481
triples_qwen_unstructured 4612.956349206349
triples_gemma_structured 539.4235638699924
triples_gemma_semi-structured 1264.75
triples_gemma_unstructured 3684.3161375661375
triples_llama_structured 1431.4481292517007
triples_llama_semi-structured 1315.281462585034
triples_llama_unstructured 2553.1366213151928


In [4]:
def extract_triples(input_string):
    """
    Extracts only 3-element tuples, accepting single or double quotes.
    """

    input_string = str(input_string)
    processed_string = input_string.replace('*', '"')

    # Quoting Group (Q): This group (r'["\']') matches either a double quote OR a single quote.
    Q = r'["\']' 
        
    # Flexible Pattern:
    pattern = rf"""
        \(              # Match the literal opening parenthesis (
        ({Q}.*?{Q})     # Group 1: Capture the first quoted string
        ,\s* # Match comma, optional whitespace
        ({Q}.*?{Q})     # Group 2: Capture the second quoted string
        ,\s* # Match comma, optional whitespace
        ({Q}.*?{Q})     # Group 3: Capture the third quoted string
        \)              # Match the literal closing parenthesis )
    """
    
    matches = re.findall(pattern, processed_string, re.VERBOSE | re.DOTALL)
    
    extracted_data = []
    for str1_quoted, str2_quoted, str3_quoted in matches:
        # Remove the surrounding quotes from each captured string
        # using the replace method, which handles both ' and "
        str1 = str1_quoted.strip().replace('"', '').replace("'", '')
        str2 = str2_quoted.strip().replace('"', '').replace("'", '')
        str3 = str3_quoted.strip().replace('"', '').replace("'", '')
        extracted_data.append((str1, str2, str3))
        
    return extracted_data

for model in ["qwen", "gemma", "llama"]:
    for prompt in ["structured", "semi-structured", "unstructured"]:
        df[f"triples_{model}_{prompt}"] = df[f"triples_{model}_{prompt}"].apply(extract_triples)

In [5]:
def discover_common_predicates(df, triple_columns, top_n=20):
    """
    Identifies the most frequent middle elements (index 1) across all triple columns.
    """
    all_predicates = []
    for col in triple_columns:
        # Flatten all triples and grab the middle item
        preds = [t[1] for sublist in df[col] if isinstance(sublist, list) 
                 for t in sublist if len(t) == 3]
        all_predicates.extend(preds)
    
    # Get the most frequent items that look like relationship keys (usually uppercase)
    top_preds = pd.Series(all_predicates).value_counts().head(top_n).index.tolist()
    return set(top_preds)

# Run discovery
triple_cols = [col for col in df.columns if 'triples' in col]
discovered_preds = discover_common_predicates(df, triple_cols)

def fix_shifted_triples(triple_list, known_predicates):
    # Check if triple_list is actually a list (handles NaN/None)
    if not isinstance(triple_list, list):
        return triple_list
        
    fixed_triples = []
    for triple in triple_list:
        if not isinstance(triple, (tuple, list)) or len(triple) != 3:
            fixed_triples.append(triple)
            continue
            
        s, p, o = triple
        
        # Scenario A: Predicate is in Subject position
        if s in known_predicates and p not in known_predicates:
            fixed_triples.append((p, s, o))
        # Scenario B: Predicate is in Object position
        elif o in known_predicates and p not in known_predicates:
            fixed_triples.append((s, o, p))
        else:
            fixed_triples.append(triple)
            
    return fixed_triples

# Apply the fix using discovered predicates
for col in triple_cols:
    df[col] = df[col].apply(lambda x: fix_shifted_triples(x, discovered_preds))

In [6]:
# Convert string representations of lists into actual lists
triple_cols = [col for col in df.columns if 'triples' in col]

for col in triple_cols:
    # Using ast.literal_eval is safer than eval()
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Calculate average number of triples per row for each model
avg_triples = df[triple_cols].map(len).mean()

print("Average Number of Triples per Column:")
print(avg_triples)

total_triples = df[triple_cols].map(len).sum()
print("\nTotal Number of Triples per Column:")
print(total_triples)

Average Number of Triples per Column:
triples_qwen_structured          15.721088
triples_qwen_semi-structured     17.481387
triples_qwen_unstructured        49.505574
triples_gemma_structured          7.950586
triples_gemma_semi-structured    17.177438
triples_gemma_unstructured       45.018707
triples_llama_structured         20.653439
triples_llama_semi-structured    19.132937
triples_llama_unstructured       31.117347
dtype: float64

Total Number of Triples per Column:
triples_qwen_structured          166392
triples_qwen_semi-structured     185023
triples_qwen_unstructured        523967
triples_gemma_structured          84149
triples_gemma_semi-structured    181806
triples_gemma_unstructured       476478
triples_llama_structured         218596
triples_llama_semi-structured    202503
triples_llama_unstructured       329346
dtype: int64


In [7]:
avg_triples = df[triple_cols].map(str).map(len).mean()

print("Average String Length of Triples per Column:")
print(avg_triples)

Average String Length of Triples per Column:
triples_qwen_structured          1200.935752
triples_qwen_semi-structured     1376.460884
triples_qwen_unstructured        4613.161659
triples_gemma_structured          539.444444
triples_gemma_semi-structured    1264.790249
triples_gemma_unstructured       3684.689342
triples_llama_structured         1431.826342
triples_llama_semi-structured    1315.400227
triples_llama_unstructured       2553.103269
dtype: float64


In [8]:
# Ensure columns are lists of tuples
triple_cols = [col for col in df.columns if 'triples' in col]
for col in triple_cols:
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

def get_triple_stats(series):
    # Explode the list of triples into individual rows
    exploded = series.explode().dropna()
    
    # Calculate lengths of each component
    stats = pd.DataFrame(exploded.tolist(), columns=['subject', 'predicate', 'object'])
    
    char_lengths = {
        'avg_subj_len': stats['subject'].str.len().mean(),
        'avg_pred_len': stats['predicate'].str.len().mean(),
        'avg_obj_len': stats['object'].str.len().mean()
    }
    return char_lengths

# Apply to all triple columns
component_lengths = {col: get_triple_stats(df[col]) for col in triple_cols}
print(pd.DataFrame(component_lengths).T)

                               avg_subj_len  avg_pred_len  avg_obj_len
triples_qwen_structured           29.496827     15.868101    17.019592
triples_qwen_semi-structured      30.792685     15.914757    18.028775
triples_qwen_unstructured         36.776801     12.345174    30.051465
triples_gemma_structured          24.832060     14.854223    14.160632
triples_gemma_semi-structured     29.491364     14.847942    15.288659
triples_gemma_unstructured        38.573281     14.090073    15.174338
triples_llama_structured          21.809603     16.870949    16.627235
triples_llama_semi-structured     22.076345     16.361624    16.305758
triples_llama_unstructured        25.599294     13.954109    28.491784


In [9]:
def get_top_predicates(df, column, n=15):
    # Flatten the list of triples and extract the 2nd item (index 1)
    predicates = [t[1] for sublist in df[column] for t in sublist if len(t) == 3]
    return pd.Series(predicates).value_counts().head(n)

for model in ["qwen", "gemma", "llama"]:
    for prompt in ["structured", "semi-structured", "unstructured"]:
        print(f"Top {model} {prompt} Predicates:")
        print(get_top_predicates(df, f"triples_{model}_{prompt}"))
        print()

Top qwen structured Predicates:
INVOLVES_TASK                30505
REQUIRES_SKILL               29338
REQUIRES_QUALITY             27705
REQUIRES_EXPERIENCE_LEVEL     9874
IS_IN_INDUSTRY                9773
HAS_CONTRACT_TYPE             9119
REQUIRES_WORK_EXPERIENCE      8969
HAS_LOCATION                  7437
REQUIRES_LANGUAGE             7018
REQUIRES_EDUCATION            6077
IS_MANAGER_LEVEL              3935
HAS_SALARY_RANGE              3355
DESIRES                       2991
OFFERS_POSITION               1529
000                           1061
Name: count, dtype: int64

Top qwen semi-structured Predicates:
INVOLVES_TASK                38385
REQUIRES_SKILL               34700
REQUIRES_QUALITY             23756
REQUIRES_WORK_EXPERIENCE     14366
IS_IN_INDUSTRY               14237
REQUIRES_EXPERIENCE_LEVEL    11011
HAS_CONTRACT_TYPE             8552
DESIRES                       8435
REQUIRES_LANGUAGE             7897
REQUIRES_EDUCATION            7454
IS_MANAGER_LEVEL             

In [10]:
def get_top_entities(df, column, part_idx=0, n=10):
    # part_idx=0 for Subject, part_idx=2 for Object
    entities = [t[part_idx] for sublist in df[column] for t in sublist if len(t) == 3]
    return pd.Series(entities).value_counts().head(n)

print("Most Frequent Subjects (Entities):")
print(get_top_entities(df, 'triples_qwen_structured', part_idx=0))

Most Frequent Subjects (Entities):
Denmark                 1485
Lagermedarbejder         808
Financial Controller     653
Copenhagen               653
Projektleder             616
Pædagog                  585
Servicetekniker          547
Salgskonsulent           541
Bogholder                498
false                    490
Name: count, dtype: int64


In [11]:
def get_diversity(series):
    flat_triples = [str(t) for sublist in series for t in sublist]
    if not flat_triples: return 0
    return len(set(flat_triples)) / len(flat_triples)

diversity_scores = {col: get_diversity(df[col]) for col in triple_cols}
print("Triple Diversity (Unique / Total):")
print(pd.Series(diversity_scores))

Triple Diversity (Unique / Total):
triples_qwen_structured          0.885535
triples_qwen_semi-structured     0.882063
triples_qwen_unstructured        0.964128
triples_gemma_structured         0.871930
triples_gemma_semi-structured    0.889525
triples_gemma_unstructured       0.969319
triples_llama_structured         0.860240
triples_llama_semi-structured    0.820659
triples_llama_unstructured       0.975658
dtype: float64


In [12]:
# df.to_excel("./outputs/clean_outputs/filtered_triples.xlsx")

In [13]:
df_int = pd.read_csv("../dataset/final_dataset/contacted_anon.csv")
df_int.head()

,humanjobid,response,cvid
0,1527545.0,NaN,971b2f2bbccb436dbf11215a8e5c9463
1,1527545.0,no4,6652b22903a84153903b7e5fef960897
2,1527545.0,NaN,6a21e3c95b364fda8633413972e936f4
3,1527545.0,NaN,ecc450aa96f3489b9ade93a3a5c13423
4,1527545.0,no4,c5692d7b3df74c1880b9f228fe6ed115


In [20]:
df_int["response"].fillna(0).value_counts().sum()

242167

In [18]:
texts = defaultdict(list)

with open("../dataset/final_dataset/jobs.json", 'r') as f:
    data = json.load(f)

for item in data:
    texts["id"].append(item.get("humanjobid", ""))
    texts["company"].append(item.get("companytext", ""))
    texts["job title"].append(item.get("jobtitle", ""))
    texts["text"].append(item.get("searchtext", ""))
 
df = pd.DataFrame(texts)
df.head()

,id,company,job title,text
0,1527392,Jobindex,"IT-administrator – få indflydelse på et setup,...",Vil du ind i en virksomhed i rivende udvikling...
1,1527395,Jobindex,"IT-administrator – få indflydelse på et setup,...","IT-administrator – få indflydelse på et setup,..."
2,1527397,Aqua d'Or Mineral Water A/S,SQE Manager,For jobsøgere For arbejdsgivere Aqua d'Or Mi...
3,1527417,Klimabrands,Kundeservice / teknisk support,For jobsøgere For arbejdsgivere mailto:job@k...
4,1527440,Scan Studio ApS,Retail designer med teknikken på plads,For jobsøgere For arbejdsgivere Scan Studio ...


In [19]:
df.shape

(10584, 4)

In [24]:
df_CVs = pd.read_excel("./outputs/final_outputs/cv.xlsx")
df_CVs.shape

(83804, 3)